In [1]:
from FCN_model import FCN
import torch
import torch.optim as optim
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np
import pandas as pd
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
    torch.cuda.manual_seed_all(42)

In [2]:
origin_FCN = FCN(in_channels = 1, num_classes=2).to("cuda")

In [3]:
origin_FCN

FCN(
  (layers): Sequential(
    (0): Conv1d(1, 128, kernel_size=(8,), stride=(1,), padding=same)
    (1): BatchNorm1d(128, eps=0.001, momentum=0.99, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv1d(128, 256, kernel_size=(5,), stride=(1,), padding=same)
    (4): BatchNorm1d(256, eps=0.001, momentum=0.99, affine=True, bias=True, track_running_stats=True)
    (5): ReLU()
    (6): Conv1d(256, 128, kernel_size=(3,), stride=(1,), padding=same)
    (7): BatchNorm1d(128, eps=0.001, momentum=0.99, affine=True, bias=True, track_running_stats=True)
    (8): ReLU()
    (9): AdaptiveAvgPool1d(output_size=1)
    (10): Flatten(start_dim=1, end_dim=-1)
    (11): Linear(in_features=128, out_features=2, bias=True)
  )
)

In [4]:
epochs = 2000
optimizer = torch.optim.Adam(origin_FCN.parameters(), lr=0.001)

criterion = nn.CrossEntropyLoss()

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=50,
    min_lr=0.0001
)

#### Test with original dataset without z-score again

In [5]:
FordA_train = np.load("FordA_processed/FordA_TRAIN.npy")
FordA_test = np.load("FordA_processed/FordA_TEST.npy")

In [6]:
FordA_X_train = np.expand_dims(FordA_train[:, 1:], axis=1)
FordA_y_train = FordA_train[:, 0]

In [7]:
print(FordA_X_train.shape)
print(FordA_y_train.shape)

(3601, 1, 500)
(3601,)


In [8]:
FordA_y_train = (FordA_y_train == 1).astype(np.int64)


In [9]:
FordA_X_test = np.expand_dims(FordA_test[:, 1:], axis=1)
FordA_y_test = FordA_test[:, 0]
FordA_y_test = (FordA_y_test == 1).astype(np.int64)

In [10]:
FordA_X_train = torch.tensor(FordA_X_train, dtype=torch.float32)
FordA_y_train = torch.tensor(FordA_y_train, dtype=torch.long)

In [11]:
FordA_X_test = torch.tensor(FordA_X_test, dtype=torch.float32)
FordA_y_test = torch.tensor(FordA_y_test, dtype=torch.long)

In [12]:
dataset = TensorDataset(FordA_X_train, FordA_y_train)

train_loader = DataLoader(
    dataset,
    batch_size=360, # calculated as FordA_X_train.shape[0] / 10, according to tensorflow version
    shuffle=True
)

#### Training experiment 1

In [13]:
metrics = {
    "epoch": [],
    "loss": [],
    "accuracy": [],
    "precision": [],
    "recall": [],
    "f1": []
}

In [14]:
for epoch in range(epochs):
    metrics["epoch"].append(epoch)
    origin_FCN.train()
    running_loss = 0.0
    total_samples = 0
    all_predictions = []
    all_labels = []

    for X, y in train_loader:
        X = X.to("cuda")
        y = y.to("cuda")

        optimizer.zero_grad()
        # print(batch_features.shape)
        # Forward pass
        outputs = origin_FCN(X)
        loss = criterion(outputs, y)

        # print(outputs)
        # print(loss)

         # Backward pass and optimization
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * X.size(0)
        total_samples += X.size(0)

        predictions = outputs.argmax(dim=1)
        all_predictions.extend(
            predictions.detach().cpu().numpy()
        )
        all_labels.extend(
            y.cpu().numpy()
        )
    epoch_loss = running_loss / total_samples

    accuracy = accuracy_score(all_labels, all_predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
                                                        all_labels,
                                                        all_predictions,
                                                        average="macro",
                                                        zero_division=0
                                                    )
    # Save metrics
        
    metrics["loss"].append(epoch_loss)
    metrics["accuracy"].append(accuracy)
    metrics["precision"].append(precision)
    metrics["recall"].append(recall)
    metrics["f1"].append(f1)

    print(
            f"Epoch {epoch+1}/{epochs} | "
            f"Loss: {epoch_loss:.4f} | "
            f"Accuracy: {accuracy:.4f} | "
            f"Precision: {precision:.4f} | "
            f"Recall: {recall:.4f} | "
            f"F1: {f1:.4f}"
    )
    #     break
    # break
metrics_df = pd.DataFrame(metrics)
metrics_df.to_csv("FCN_original_metrics.csv")

c:\Users\96561\miniconda3\envs\UCR-pytorch\Lib\site-packages\torch\nn\modules\conv.py:380: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\native\Convolution.cpp:1105.)
  return F.conv1d(


Epoch 1/2000 | Loss: 0.5498 | Accuracy: 0.7148 | Precision: 0.7560 | Recall: 0.7197 | F1: 0.7057
Epoch 2/2000 | Loss: 0.4443 | Accuracy: 0.7992 | Precision: 0.7994 | Recall: 0.7988 | F1: 0.7989
Epoch 3/2000 | Loss: 0.3962 | Accuracy: 0.8353 | Precision: 0.8353 | Recall: 0.8355 | F1: 0.8353
Epoch 4/2000 | Loss: 0.3821 | Accuracy: 0.8375 | Precision: 0.8387 | Recall: 0.8383 | F1: 0.8375
Epoch 5/2000 | Loss: 0.3704 | Accuracy: 0.8389 | Precision: 0.8395 | Recall: 0.8395 | F1: 0.8389
Epoch 6/2000 | Loss: 0.3439 | Accuracy: 0.8667 | Precision: 0.8679 | Recall: 0.8660 | F1: 0.8664
Epoch 7/2000 | Loss: 0.3321 | Accuracy: 0.8761 | Precision: 0.8771 | Recall: 0.8755 | F1: 0.8759
Epoch 8/2000 | Loss: 0.3159 | Accuracy: 0.8814 | Precision: 0.8823 | Recall: 0.8808 | F1: 0.8812
Epoch 9/2000 | Loss: 0.3043 | Accuracy: 0.8870 | Precision: 0.8877 | Recall: 0.8864 | F1: 0.8868
Epoch 10/2000 | Loss: 0.3010 | Accuracy: 0.8870 | Precision: 0.8869 | Recall: 0.8871 | F1: 0.8869
Epoch 11/2000 | Loss: 0.2929 

In [16]:
metrics_df.sort_values(by="accuracy", ascending=False)

,epoch,loss,accuracy,precision,recall,f1
1897,1897,0.119426,0.960289,0.960203,0.960397,0.960273
1889,1889,0.123700,0.959733,0.959666,0.959771,0.959713
1983,1983,0.118549,0.959456,0.959430,0.959430,0.959430
1980,1980,0.121243,0.958623,0.958678,0.958519,0.958589
1905,1905,0.122286,0.958345,0.958297,0.958346,0.958321
...,...,...,...,...,...,...
4,4,0.370445,0.838934,0.839520,0.839491,0.838934
3,3,0.382093,0.837545,0.838670,0.838291,0.837534
2,2,0.396163,0.835324,0.835307,0.835520,0.835294
1,1,0.444304,0.799222,0.799363,0.798764,0.798927
